### General Agent

In [4]:
!pip install -q openai python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import os
from getpass import getpass
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

In [5]:
from openai import OpenAI
client = OpenAI()

In [6]:
def calculator(expression: str) -> str:
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

In [7]:
SYSTEM_PROMPT = """
You are a helpful AI agent.

You can:
- Answer general questions
- Use a calculator tool when math is required

If math is required:
Respond in JSON format:
{
  "tool": "calculator",
  "input": "expression here"
}

Otherwise:
Respond normally with text.
"""

In [8]:
import json

def run_agent(user_query):

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_query}
        ]
    )

    message = response.choices[0].message.content

    # Try parsing as tool call
    try:
        parsed = json.loads(message)

        if parsed.get("tool") == "calculator":
            tool_result = calculator(parsed["input"])

            # Send tool result back to model
            final_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_query},
                    {"role": "assistant", "content": message},
                    {"role": "tool", "content": tool_result}
                ]
            )

            return final_response.choices[0].message.content

    except:
        return message

In [9]:
print(run_agent("What is the capital of France?"))

The capital of France is Paris.


In [10]:
print(run_agent("What is 234 * 56?"))

{
  "tool": "calculator",
  "input": "234 * 56"
}


### AGENT Builder

In [11]:
!pip install -q openai-agents
from agents import Agent, Runner, function_tool


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [12]:
@function_tool
def calculator(expression: str) -> str:
    """
    Evaluates a mathematical expression.
    Example input: "234 * 56"
    """
    try:
        result = eval(expression)
        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

In [13]:
agent = Agent(
    name="SimpleMathAgent",
    instructions="""
You are a helpful AI agent.

- Answer general knowledge questions directly.
- Use the calculator tool whenever mathematical computation is required.
- Always use the tool for arithmetic.
""",
    tools=[calculator],
    model="gpt-4o-mini"
)

In [14]:
result = await Runner.run(
    agent,
    "What is weather in chennai"
)

print(result.final_output)

I don't have real-time capabilities to check current weather. However, you can easily find the weather in Chennai by checking a weather website, app, or service for updates. Would you like to know typical weather patterns in Chennai throughout the year?


### Single Financial Analyst Agent

In [15]:
from openai import OpenAI
from agents import Agent, Runner, function_tool
from typing import Dict
import math

# -----------------------------------
# Tool Definitions
# -----------------------------------

@function_tool
def compound_interest(principal: float, rate: float, years: int) -> Dict:
    """
    Calculate compound interest.
    """
    amount = principal * ((1 + rate) ** years)
    return {
        "principal": principal,
        "rate": rate,
        "years": years,
        "final_amount": round(amount, 2)
    }

@function_tool
def risk_score(volatility: float, beta: float) -> Dict:
    """
    Simple risk scoring.
    """
    score = volatility * beta
    return {
        "volatility": volatility,
        "beta": beta,
        "risk_score": round(score, 3)
    }

# -----------------------------------
# Single Agent
# -----------------------------------

financial_agent = Agent(
    name="Financial Analyst Agent",
    instructions="""
    You are a financial analyst.
    Use tools whenever calculations are required.
    Explain results clearly.
    """,
    tools=[compound_interest, risk_score],
    model="gpt-4o-mini"
)

# -----------------------------------
# Execution
# -----------------------------------

result = await Runner.run(
    financial_agent,
    """
    If I invest 10000 at 8% for 5 years,
    and volatility is 0.2 and beta is 1.3,
    calculate final amount and risk score.
    """
)

print(result.final_output)

### Investment Calculation

If you invest **$10,000** at an interest rate of **8%** for **5 years**, the final amount is:

- **Final Amount:** **$59,049.00**

### Risk Score Calculation

With a volatility of **0.2** and a beta of **1.3**, the calculated risk score is:

- **Risk Score:** **0.26**

### Summary
- After 5 years, your investment will grow to **$59,049.00**.
- The risk score of **0.26** indicates a moderate level of risk associated with the investment, taking into account its volatility and beta.


### Multi-Agent Architecture

In [33]:
from agents import Agent, Runner, function_tool
import asyncio

# -----------------------------------
# Tools
# -----------------------------------

@function_tool
def compound_interest(principal: float, rate: float, years: int):
    amount = principal * ((1 + rate) ** years)
    return {"final_amount": round(amount, 2)}

@function_tool
def risk_score(volatility: float, beta: float):
    score = volatility * beta
    return {"risk_score": round(score, 3)}

# -----------------------------------
# Specialist Agent
# -----------------------------------

calculation_agent = Agent(
    name="Calculation Agent",
    instructions="""
    You only perform financial calculations.
    Always use tools.
    Return structured JSON only.
    """,
    tools=[compound_interest, risk_score],
    model="gpt-4o-mini"
)

# -----------------------------------
# Reviewer Agent
# -----------------------------------

review_agent = Agent(
    name="Review Agent",
    instructions="""
    Review calculation results.
    Explain them clearly in business language.
    Add interpretation and risk commentary.
    """,
    model="gpt-4o-mini"
)

# -----------------------------------
# Planner Agent (Orchestrator)
# -----------------------------------

planner_agent = Agent(
    name="Planner Agent",
    instructions="""
    Break user request into:
    1. Calculation task
    2. Review task

    First delegate to Calculation Agent.
    Then send its structured output to Review Agent.
    """,
    handoffs=[calculation_agent, review_agent],
    model="gpt-4o-mini"
)

# -----------------------------------
# Execution
# -----------------------------------



In [34]:
result = await Runner.run(
    planner_agent,
    """
    If I invest 10000 at 8% for 5 years,
    and volatility is 0.2 and beta is 1.3,
    calculate final amount and risk score.
    """
)

print(result.final_output)

Tool name 'transfer_to_Calculation Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_calculation_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Review Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_review_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Calculation Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_calculation_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.
Tool name 'transfer_to_Review Agent' contains invalid characters for function calling and has been transformed to 'transfer_to_review_agent'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


{
  "final_amount": 14693.28,
  "risk_score": 0.26
}


### CRAG

In [20]:
!pip install -q langchain langchain-openai langchain-community langchain-text-splitters faiss-cpu tiktoken


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [21]:
import os
from getpass import getpass

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="gpt-4o-mini",
)

In [23]:
documents = [
    Document(page_content="""
    Standard RAG retrieves documents and generates answers from them.
    It may still hallucinate if retrieval is weak.
    """),

    Document(page_content="""
    Corrective RAG evaluates retrieval quality.
    If documents are insufficient, it rewrites the query and re-retrieves.
    """),

    Document(page_content="""
    Retrieval evaluation improves factual reliability in enterprise systems.
    """)
]

In [30]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    
)

docs = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

PermissionDeniedError: Error code: 403 - {'error': {'message': 'Project `proj_UlkXwqGJYmangzQGp2y4vHX4` does not have access to model `text-embedding-ada-002`', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

In [35]:
evaluation_prompt = ChatPromptTemplate.from_template("""
You are a retrieval evaluator.

Question:
{question}

Retrieved Context:
{context}

Is the context sufficient to answer the question?

Respond ONLY with:
SUFFICIENT
or
INSUFFICIENT
""")

In [37]:
rewrite_prompt = ChatPromptTemplate.from_template("""
Rewrite this question to improve document retrieval precision.

Original Question:
{question}
""")

In [38]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer strictly using the context below.

Question:
{question}

Context:
{context}

If insufficient information exists, say:
"I do not have enough information."
""")

In [39]:
evaluation_chain = evaluation_prompt | llm
rewrite_chain = rewrite_prompt | llm
answer_chain = answer_prompt | llm

In [40]:
class CRAGState:
    def __init__(self, question):
        self.original_question = question
        self.current_question = question
        self.context = ""
        self.retrieval_evaluation = None
        self.rewritten = False

In [41]:
def retrieve_step(state: CRAGState):
    docs = retriever.invoke(state.current_question)
    state.context = "\n\n".join([d.page_content for d in docs])
    return state

In [42]:
def rewrite_step(state: CRAGState):

    if "INSUFFICIENT" in state.retrieval_evaluation:

        result = rewrite_chain.invoke({
            "question": state.original_question
        })

        state.current_question = result.content.strip()
        state.rewritten = True

        # Re-retrieve
        retrieve_step(state)

    return state

In [43]:
def generate_answer_step(state: CRAGState):

    result = answer_chain.invoke({
        "question": state.original_question,
        "context": state.context
    })

    return result.content

In [44]:
def corrective_rag_pipeline(question: str):

    state = CRAGState(question)

    # Initial retrieval
    retrieve_step(state)

    # Evaluate retrieval
    evaluate_step(state)

    print("Retrieval Evaluation:", state.retrieval_evaluation)

    # Correct if needed
    rewrite_step(state)

    if state.rewritten:
        print("Query was rewritten to:", state.current_question)

    # Final answer
    answer = generate_answer_step(state)

    return answer

In [45]:
response = corrective_rag_pipeline(
    "How does corrective RAG improve reliability?"
)

print("\nFinal Answer:\n", response)

NameError: name 'retriever' is not defined